In [ ]:
# =============================================================================
# TRAIN TRIAL HEAD — Standalone Training Notebook
# =============================================================================
# Mục đích: Train 2 MLP nhỏ (GatingMLP + RelationMLP) theo paper TRIAL
#           trên tập MMDocIR training data, lưu weights ra file.
#
# Output:   /kaggle/working/trial_head_weights.pt  (~vài MB)
#           → Tải về, upload lên Kaggle dataset
#           → Dùng trong connect.ipynb (eval pipeline) để score
#
# Chạy theo thứ tự: BƯỚC 0 → 1 → 2 → 3 → 3.5 → 3.6
# =============================================================================

# --- BƯỚC 0: SETUP ---
!pip uninstall -y tensorflow pyarrow
!pip install "pyarrow<20.0.0"
!pip install -U -q git+https://github.com/illuin-tech/colpali
!pip install -U -q transformers accelerate peft bitsandbytes

import torch
import warnings
import os
import gc
warnings.filterwarnings('ignore')


In [ ]:

 # --- BƯỚC 1: LOAD TRAINING DATA (LOW-MEM, LAZY PARQUET) ---
import os, gc, random
import pandas as pd
import pyarrow.parquet as pq

TRAIN_PARQUET_DIR = "/kaggle/input/datasets/nguyenducdung1107/training-dataset-mmdocir/MMDOC_TRAINING/parquet"
TOP1_NEG_DIR      = "/kaggle/input/datasets/nguyenducdung1107/training-dataset-mmdocir/MMDOC_TRAINING/top1_neg"

ALL_DATASETS = ["SlideVQA", "ArxivQA", "MP-DocVQA", "TAT-DQA", "SciQAG", "Wiki-ss", "DUDE"]

# Chạy tuần tự từng domain để tránh crash RAM, nhưng vẫn phủ 0.5 toàn bộ tập sau nhiều lần run.
# Chỉ cần đổi DOMAIN_IDX (0..len(ALL_DATASETS)-1) thay vì sửa tên domain thủ công.
DOMAIN_IDX = 0
if DOMAIN_IDX < 0 or DOMAIN_IDX >= len(ALL_DATASETS):
    raise ValueError(f"DOMAIN_IDX must be in [0, {len(ALL_DATASETS)-1}], got {DOMAIN_IDX}")

TARGET_DOMAIN = ALL_DATASETS[DOMAIN_IDX]
SELECTED_DATASETS = [TARGET_DOMAIN]

SAMPLE_FRAC = 0.50  # tăng lên 0.5 theo yêu cầu
MAX_ROWS_PER_DATASET = None  # None = dùng toàn bộ 50% của domain
MAX_TOTAL_PAGES = None       # None = không cắt tổng pages của domain
RANDOM_SEED = 42

print("=" * 60)
print(f" BƯỚC 1: Loading Training Parquet Files (sample {SAMPLE_FRAC*100:.0f}%)...")
print(f" DOMAIN_IDX: {DOMAIN_IDX} | TARGET_DOMAIN: {TARGET_DOMAIN}")
print(f" Domains: {SELECTED_DATASETS}")
print("=" * 60)

META_COLS = ["file_name", "page"]


def build_lazy_sample_index(path, dataset_name, frac=0.10, seed=42, max_rows=None):
    pf = pq.ParquetFile(path)
    total_rows = pf.metadata.num_rows
    target_rows = max(1, int(total_rows * frac))
    if max_rows is not None:
        target_rows = min(target_rows, max_rows)

    group_ids = list(range(pf.num_row_groups))
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    sampled_frames = []
    selected_rows = 0

    for rg in group_ids:
        if selected_rows >= target_rows:
            break

        table = pf.read_row_group(rg, columns=META_COLS)
        if table.num_rows == 0:
            del table
            continue

        part = table.to_pandas().reset_index(drop=True)
        del table

        part["row_group_id"] = rg
        part["row_in_group"] = range(len(part))
        rows_left = target_rows - selected_rows
        if len(part) > rows_left:
            part = part.sample(n=rows_left, random_state=seed + rg).reset_index(drop=True)

        part["dataset"] = dataset_name
        part["source_parquet"] = path
        sampled_frames.append(part)
        selected_rows += len(part)
        gc.collect()

    if not sampled_frames:
        return pd.DataFrame(columns=META_COLS + ["row_group_id", "row_in_group", "dataset", "source_parquet"]), total_rows

    out = pd.concat(sampled_frames, ignore_index=True)
    del sampled_frames
    gc.collect()
    return out, total_rows


dfs = []
for name in SELECTED_DATASETS:
    path = os.path.join(TRAIN_PARQUET_DIR, f"{name}_filter.parquet")
    if not os.path.exists(path):
        print(f"   [SKIP] {name}: file not found.")
        continue

    try:
        df_tmp, n_full = build_lazy_sample_index(
            path,
            dataset_name=name,
            frac=SAMPLE_FRAC,
            seed=RANDOM_SEED,
            max_rows=MAX_ROWS_PER_DATASET,
        )
    except Exception as e:
        print(f"   [SKIP] {name}: read failed: {e}")
        continue

    dfs.append(df_tmp)
    print(f"   {name}: {n_full} total -> sampled {len(df_tmp)}")

    del df_tmp
    gc.collect()

if not dfs:
    raise RuntimeError("No dataset loaded. Check parquet path or permissions.")

sample_layouts_df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
del dfs
gc.collect()

if MAX_TOTAL_PAGES is not None and len(sample_layouts_df) > MAX_TOTAL_PAGES:
    sample_layouts_df = sample_layouts_df.sample(n=MAX_TOTAL_PAGES, random_state=RANDOM_SEED).reset_index(drop=True)

sample_layouts_df["join_doc_name"] = (
    sample_layouts_df["dataset"] + "__" + sample_layouts_df["file_name"].astype(str)
)

# Chỉ giữ metadata nhẹ; image/layouts sẽ được đọc lazy ở BƯỚC 3
sample_layouts_df["final_text"] = ""
sample_layouts_df["img_type"] = "lazy_parquet"
sample_layouts_df["img_data"] = None

keep_cols = [
    "dataset", "file_name", "join_doc_name", "page",
    "source_parquet", "row_group_id", "row_in_group",
    "final_text", "img_type", "img_data"
]
sample_layouts_df = sample_layouts_df[keep_cols].copy()

gc.collect()

print(f"\n Total pages: {len(sample_layouts_df)}")
print(f" Datasets: {sample_layouts_df['dataset'].value_counts().to_dict()}")
print(" LOW-MEM LAZY mode enabled: metadata only in RAM, heavy columns loaded in BƯỚC 3")


In [ ]:

# --- BƯỚC 2: LOAD MODEL ---
import torch, gc
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "vidore/colSmol-500M"

model = ColIdefics3.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="eager"
).eval()
processor = ColIdefics3Processor.from_pretrained(MODEL_NAME)

print(f" Model ready on {device}!")


In [ ]:
# --- BƯỚC 3: FUSED INDEXING (Training Pages) ---
# Copy cell này thay thế BƯỚC 3 cũ trong notebook
# Progress bar hiển thị: pages đã encode, tốc độ (pg/s), ETA

import io, pickle, torch, gc, os, time
import pyarrow.parquet as pq
from PIL import Image
from tqdm.notebook import tqdm

FINAL_INDEX_PATH = os.path.join(WORKING_DIR, "colsmol_fused_index.pkl")
CHECKPOINT_PATH  = os.path.join(WORKING_DIR, "colsmol_checkpoint.pkl")
BATCH_SIZE = 32    # RTX PRO 6000: 95GB VRAM → batch lớn, tận dụng GPU
SAVE_EVERY = 500

MAX_VIS_TOKENS = 96
MAX_TXT_TOKENS = 64

fused_index = []
idx_map     = []
start_idx = 0
skip_encoding = False

if os.path.exists(FINAL_INDEX_PATH):
    with open(FINAL_INDEX_PATH, 'rb') as f:
        saved = pickle.load(f)
    fused_index = saved['fused_index']
    idx_map     = saved['idx_map']
    print(f" Loaded {len(fused_index)} existing embeddings. Skipping encode.")
    skip_encoding = True
elif os.path.exists(CHECKPOINT_PATH):
    try:
        with open(CHECKPOINT_PATH, 'rb') as f:
            saved = pickle.load(f)
        fused_index = saved['fused_index']
        idx_map     = saved['idx_map']
        start_idx   = saved['next_row']
        print(f" Resuming from checkpoint: {len(fused_index)} done, continue from row {start_idx}.")
    except:
        fused_index, idx_map, start_idx = [], [], 0
else:
    print(" Starting fresh encoding...")

if not skip_encoding:
    remaining_df = sample_layouts_df.iloc[start_idx:].copy().reset_index(drop=True)
    remaining_df["row_pos"] = range(start_idx, start_idx + len(remaining_df))
    remaining_df = remaining_df.sort_values(["source_parquet", "row_group_id", "row_in_group"]).reset_index(drop=True)

    total_pages = len(remaining_df)
    print(f" Pages to encode: {total_pages}")
    print(f" Batch size: {BATCH_SIZE} | Estimated batches: {(total_pages + BATCH_SIZE - 1) // BATCH_SIZE}")

    parquet_cache = {}

    def get_parquet_file(path):
        pf = parquet_cache.get(path)
        if pf is None:
            pf = pq.ParquetFile(path)
            parquet_cache[path] = pf
        return pf

    def build_text_from_layouts(layouts):
        try:
            lay_list = list(layouts) if isinstance(layouts, (list, tuple)) else list(layouts)
        except:
            return "Document page."
        if not lay_list:
            return "Document page."
        text_keys = ["text", "ocr_text", "content", "caption", "value"]
        texts = []
        for lay in lay_list:
            if not isinstance(lay, dict):
                continue
            for tk in text_keys:
                v = lay.get(tk)
                if v and isinstance(v, str) and len(v.strip()) > 2:
                    texts.append(v.strip())
                    break
        if texts:
            return " ".join(texts)[:1000]
        return "Document page."

    def compact_fused(vis_emb, txt_emb):
        vis_small = vis_emb[:MAX_VIS_TOKENS]
        txt_small = txt_emb[:MAX_TXT_TOKENS]
        fused = torch.cat([vis_small, txt_small], dim=0)
        return fused.to(torch.float16).contiguous()

    batch_imgs, batch_texts, batch_rows = [], [], []
    processed = 0
    n_batches_done = 0
    t0 = time.time()

    # ── Progress bar theo PAGES (không phải row groups) ──
    pbar = tqdm(total=total_pages, desc="Encoding", unit="pg")

    grouped = remaining_df.groupby(["source_parquet", "row_group_id"], sort=False)
    for (source_path, rg), group_df in grouped:
        try:
            pf = get_parquet_file(source_path)
            table = pf.read_row_group(int(rg), columns=["layouts", "image"])
            rg_data = table.to_pydict()
            del table
        except Exception as e:
            print(f" Read error ({source_path}, rg={rg}): {e}. Skip.")
            continue

        layouts_col = rg_data.get("layouts", [])
        image_col = rg_data.get("image", [])

        for _, row in group_df.iterrows():
            row_in_group = int(row["row_in_group"])
            row_pos = int(row["row_pos"])

            try:
                layouts = layouts_col[row_in_group] if row_in_group < len(layouts_col) else None
                image_obj = image_col[row_in_group] if row_in_group < len(image_col) else None
                raw = image_obj if isinstance(image_obj, bytes) else (image_obj.get("bytes") if isinstance(image_obj, dict) else None)

                if raw:
                    img = Image.open(io.BytesIO(raw)).convert("RGB")
                    if img.width < 14 or img.height < 14:
                        img = Image.new("RGB", (224, 224), "white")
                else:
                    img = Image.new("RGB", (224, 224), "white")

                text = build_text_from_layouts(layouts)
            except Exception:
                img = Image.new("RGB", (224, 224), "white")
                text = "Document page."

            batch_imgs.append(img)
            batch_texts.append(str(text).replace("<image>", " ")[:1000])
            batch_rows.append(row_pos)
            processed += 1

            if len(batch_imgs) >= BATCH_SIZE:
                with torch.no_grad():
                    try:
                        vis_inp = processor.process_images(batch_imgs).to(device)
                        out_vis = model(**vis_inp)
                        txt_inp = processor.process_queries(batch_texts, max_length=192, suffix="").to(device)
                        out_txt = model(**txt_inp)

                        for k in range(len(batch_imgs)):
                            fused_index.append(compact_fused(out_vis[k].cpu(), out_txt[k].cpu()))
                            idx_map.append(batch_rows[k])

                        del vis_inp, txt_inp, out_vis, out_txt
                    except Exception as e:
                        print(f" Encode batch error near processed={processed}: {e}. Skipping.")

                # ── Update progress bar ──
                n_batches_done += 1
                pbar.update(len(batch_imgs))
                elapsed = time.time() - t0
                speed = len(fused_index) / elapsed if elapsed > 0 else 0
                pbar.set_postfix({
                    "done": len(fused_index),
                    "pg/s": f"{speed:.1f}",
                    "elapsed": f"{elapsed/60:.1f}m",
                })

                batch_imgs, batch_texts, batch_rows = [], [], []
                torch.cuda.empty_cache()
                gc.collect()

                if len(fused_index) > 0 and len(fused_index) % SAVE_EVERY == 0:
                    with open(CHECKPOINT_PATH, 'wb') as f:
                        pickle.dump({'fused_index': fused_index, 'idx_map': idx_map, 'next_row': processed + start_idx}, f)
                    pbar.write(f"  💾 Checkpoint saved: {len(fused_index)} pages")

        del rg_data, layouts_col, image_col
        gc.collect()

    # Flush batch cuối
    if batch_imgs:
        with torch.no_grad():
            try:
                vis_inp = processor.process_images(batch_imgs).to(device)
                out_vis = model(**vis_inp)
                txt_inp = processor.process_queries(batch_texts, max_length=192, suffix="").to(device)
                out_txt = model(**txt_inp)
                for k in range(len(batch_imgs)):
                    fused_index.append(compact_fused(out_vis[k].cpu(), out_txt[k].cpu()))
                    idx_map.append(batch_rows[k])
                del vis_inp, txt_inp, out_vis, out_txt
            except Exception as e:
                print(f" Final batch error: {e}.")
        pbar.update(len(batch_imgs))

    pbar.close()
    parquet_cache.clear()

    with open(FINAL_INDEX_PATH, 'wb') as f:
        pickle.dump({'fused_index': fused_index, 'idx_map': idx_map}, f)

    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)

    total_time = time.time() - t0
    avg_speed = len(fused_index) / total_time if total_time > 0 else 0
    print(f"\n{'='*60}")
    print(f" ✅ ENCODING DONE")
    print(f" Total pages: {len(fused_index)}")
    print(f" Total time:  {total_time/60:.1f} min ({total_time:.0f}s)")
    print(f" Avg speed:   {avg_speed:.1f} pages/s")
    print(f" Batch size:  {BATCH_SIZE}")
    print(f"{'='*60}")

print(f" Fused index: {len(fused_index)} entries")
print(f" Compact mode: vis<= {MAX_VIS_TOKENS}, txt<= {MAX_TXT_TOKENS}, dtype=float16")


In [ ]:
###=== CELL 6: BƯỚC 3.5 — TRIAL HEAD v2 (8 cải tiến) ===###

# Copy nội dung từ file: trial_head_v2_buoc35.py
# (Đã copy vào notebook)

import torch
import torch.nn as nn
import torch.nn.functional as F


class TrialHead(nn.Module):
    def __init__(self, embed_dim, lambda_rel=0.5, n_vis_tokens=96):
        super().__init__()
        self.lambda_rel = lambda_rel
        self.embed_dim = embed_dim
        self.n_vis_tokens = n_vis_tokens

        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.Mish(),
            nn.Linear(embed_dim, 1)
        )
        self.relation_mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

        # Fix 8: Gate bias init
        nn.init.constant_(self.gate_mlp[-1].bias, -1.0)

    def forward(self, q_embs, doc_embs_list):
        B, Nq, D = q_embs.shape
        N = len(doc_embs_list)
        device = q_embs.device

        # Fix 7: L2 normalize
        q_norm = F.normalize(q_embs, dim=-1)

        # Fix 1: Sigmoid gate
        gate_logits = self.gate_mlp(q_embs).squeeze(-1)
        q_norms = torch.norm(q_embs.detach(), dim=-1)
        valid_mask = q_norms > 1e-6
        w_q = torch.sigmoid(gate_logits)
        w_q = w_q.masked_fill(~valid_mask, 0.0)

        has_bigrams = (Nq > 1)
        if has_bigrams:
            q_bigram = torch.cat([q_embs[:, 1:], q_embs[:, :-1]], dim=-1)
            q_rel = self.relation_mlp(q_bigram)

        scores = torch.zeros(B, N, device=device)
        for d_idx in range(N):
            d_emb = doc_embs_list[d_idx]
            d_norm = F.normalize(d_emb, dim=-1)

            sim_mat = torch.matmul(q_norm, d_norm.T)
            max_sim, max_j = sim_mat.max(dim=-1)

            rel_contrib = torch.zeros(B, Nq, device=device)
            if has_bigrams:
                d_c = d_emb[max_j[:, 1:].reshape(-1)].reshape(B, Nq - 1, D)
                d_p = d_emb[max_j[:, :-1].reshape(-1)].reshape(B, Nq - 1, D)
                d_rel = self.relation_mlp(torch.cat([d_c, d_p], dim=-1))
                rel_scores = (q_rel * d_rel).sum(dim=-1)

                # Fix 4: Mask cross-modal
                j_curr = max_j[:, 1:]
                j_prev = max_j[:, :-1]
                same_modality = ((j_curr < self.n_vis_tokens) == (j_prev < self.n_vis_tokens))
                rel_contrib[:, 1:] = rel_scores * same_modality.float()

            token_scores = max_sim + self.lambda_rel * rel_contrib
            scores[:, d_idx] = (w_q * token_scores).sum(dim=-1)

        return scores, w_q

    def get_l1_reg(self, w_q):
        return w_q.abs().mean()


print("✅ TrialHead v2 defined (8 improvements)")

In [ ]:
###=== CELL 7: BƯỚC 3.6 — TRAIN TRIAL HEAD v2 (8 cải tiến) ===###

# Copy nội dung từ file: trial_head_v2_buoc36.py
# (Đã copy vào notebook)

import random, json, glob, pickle, torch, os, math
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
from torch.optim import AdamW

NUM_EPOCHS          = 20
LR                  = 5e-4
TEMPERATURE         = 0.05
NUM_RANDOM_NEGS     = 256       # RTX6000: VRAM dư → dùng nhiều negs hơn
LAMBDA_REG          = 1e-3
TRAIN_BATCH         = 8         # RTX6000: tăng batch (từ 4)
ACCUMULATION_STEPS  = 4         # eff batch = 8 × 4 = 32 (giữ nguyên)
GRAD_CLIP           = 1.0
VAL_RATIO           = 0.15

RESET_TRAIN_CHECKPOINT = False
RESET_WORKING_WEIGHTS  = False
CONTINUE_FROM_WORKING_WEIGHTS = True
LOAD_PRETRAINED_INPUT_WEIGHTS = True

TRIAL_WEIGHTS_PATH = os.path.join(WORKING_DIR, "trial_head_weights.pt")
TRIAL_CKPT_PATH    = os.path.join(WORKING_DIR, "trial_head_checkpoint.pt")
PRETRAINED_WEIGHTS_PATH = "/kaggle/input/datasets/namthi/trial-head-weights-5/trial_head_weights_5.pt"

if RESET_TRAIN_CHECKPOINT and os.path.exists(TRIAL_CKPT_PATH):
    os.remove(TRIAL_CKPT_PATH)
    print(f"⚠ Removed checkpoint: {TRIAL_CKPT_PATH}")
if RESET_WORKING_WEIGHTS and os.path.exists(TRIAL_WEIGHTS_PATH):
    os.remove(TRIAL_WEIGHTS_PATH)
    print(f"⚠ Removed working weights: {TRIAL_WEIGHTS_PATH}")

page_col = 'page' if 'page' in sample_layouts_df.columns else 'page_id'
row_to_fused = {row_idx: fused_pos for fused_pos, row_idx in enumerate(idx_map)}

page_lookup = {}
for df_row_idx, row in sample_layouts_df.iterrows():
    fused_pos = row_to_fused.get(df_row_idx)
    if fused_pos is None:
        continue
    try:
        pg = int(row[page_col])
    except:
        pg = -999
    page_lookup[(str(row['join_doc_name']), pg)] = fused_pos

print(f"✅ page_lookup: {len(page_lookup)} entries")

qa_pairs, hard_neg_map = [], {}
for jpath in glob.glob(os.path.join(TOP1_NEG_DIR, "*.jsonl")):
    ds_name = os.path.basename(jpath).replace("_train.jsonl", "")
    n_added = n_skip = 0
    with open(jpath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
            except:
                continue
            query = (rec.get("query") or "").strip()
            if not query:
                n_skip += 1
                continue
            pos_passages = rec.get("positive_passages", [])
            if not pos_passages:
                n_skip += 1
                continue
            pos_indices = []
            for pos in pos_passages:
                try:
                    idx = page_lookup.get((f"{ds_name}__{pos['doc_name']}", int(pos['page_id'])))
                except:
                    continue
                if idx is not None:
                    pos_indices.append(idx)
            if not pos_indices:
                n_skip += 1
                continue
            neg_indices = []
            for neg in rec.get("negative_passages", []):
                try:
                    idx = page_lookup.get((f"{ds_name}__{neg['doc_name']}", int(neg['page_id'])))
                except:
                    continue
                if idx is not None:
                    neg_indices.append(idx)
            qi = len(qa_pairs)
            qa_pairs.append({
                'question': query,
                'gt_indices': pos_indices,
                'domain': rec.get("domain", ds_name),
            })
            if neg_indices:
                hard_neg_map[qi] = neg_indices
            n_added += 1
    print(f"  {ds_name}: +{n_added} pairs (skipped {n_skip})")

print(f"\n✅ Total QA pairs: {len(qa_pairs)}  |  with hard neg: {len(hard_neg_map)}")

# Fix 6: Validation split
all_qa_indices = list(range(len(qa_pairs)))
random.seed(42)
random.shuffle(all_qa_indices)
n_val = max(1, int(len(all_qa_indices) * VAL_RATIO))
val_indices = set(all_qa_indices[:n_val])
train_indices = [i for i in all_qa_indices if i not in val_indices]
val_indices = list(val_indices)
print(f"✅ Train/Val split: {len(train_indices)} train / {len(val_indices)} val ({VAL_RATIO*100:.0f}%)")

D = fused_index[0].shape[-1]
N_VIS = MAX_VIS_TOKENS
trial_head = TrialHead(embed_dim=D, lambda_rel=0.5, n_vis_tokens=N_VIS).to(device)
optimizer = AdamW(trial_head.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
start_epoch, best_val_recall, skip_training = 0, -1.0, False

if os.path.exists(TRIAL_CKPT_PATH):
    ckpt = torch.load(TRIAL_CKPT_PATH, map_location=device, weights_only=False)
    trial_head.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_val_recall = ckpt.get('best_val_recall', -1.0)
    print(f"✅ Resumed from checkpoint epoch {ckpt['epoch']+1}")
elif CONTINUE_FROM_WORKING_WEIGHTS and os.path.exists(TRIAL_WEIGHTS_PATH):
    trial_head.load_state_dict(torch.load(TRIAL_WEIGHTS_PATH, map_location=device, weights_only=True))
    print(f"✅ Loaded working weights: {TRIAL_WEIGHTS_PATH}")
elif LOAD_PRETRAINED_INPUT_WEIGHTS and os.path.exists(PRETRAINED_WEIGHTS_PATH):
    trial_head.load_state_dict(torch.load(PRETRAINED_WEIGHTS_PATH, map_location=device, weights_only=True))
    print(f"✅ Loaded pretrained weights: {PRETRAINED_WEIGHTS_PATH}")
else:
    print("✅ Training from scratch.")


def pad_query_batch(query_tensors):
    max_len = max(t.shape[0] for t in query_tensors)
    dim = query_tensors[0].shape[1]
    dtype = query_tensors[0].dtype
    padded = []
    for t in query_tensors:
        if t.shape[0] == max_len:
            padded.append(t)
        else:
            pad = torch.zeros((max_len - t.shape[0], dim), dtype=dtype)
            padded.append(torch.cat([t, pad], dim=0))
    return torch.stack(padded, dim=0)


@torch.no_grad()
def evaluate_val(trial_head, val_indices, all_q_embs, fused_index,
                 qa_pairs, hard_neg_map, all_fused_indices, device,
                 num_eval_negs=200):
    trial_head.eval()
    hits_at_1 = 0
    total = 0
    for qi in val_indices:
        qa = qa_pairs[qi]
        gt_set = set(qa['gt_indices'])
        if not gt_set:
            continue
        hard_negs = set(hard_neg_map.get(qi, []))
        neg_pool = [x for x in all_fused_indices if x not in gt_set and x not in hard_negs]
        rand_negs = random.sample(neg_pool, min(num_eval_negs, len(neg_pool)))
        candidates = sorted(gt_set) + sorted(hard_negs) + rand_negs
        candidate_embs = [fused_index[c].to(device=device, dtype=torch.float32) for c in candidates]
        q_emb = all_q_embs[qi].unsqueeze(0).to(device=device, dtype=torch.float32)
        scores, _ = trial_head(q_emb, candidate_embs)
        pred_pos = scores[0].argmax().item()
        pred_idx = candidates[pred_pos]
        if pred_idx in gt_set:
            hits_at_1 += 1
        total += 1
    recall_at_1 = hits_at_1 / max(total, 1)
    trial_head.train()
    return recall_at_1


if not skip_training:
    print("✅ Encoding queries...")
    all_q_embs = []
    Q_ENCODE_BATCH = 16   # RTX6000: encode nhanh hơn với batch lớn
    with torch.no_grad():
        for i in tqdm(range(0, len(qa_pairs), Q_ENCODE_BATCH), desc="Queries"):
            batch = qa_pairs[i:i+Q_ENCODE_BATCH]
            q_inp = processor.process_queries([q['question'] for q in batch]).to(device)
            q_out = model(**q_inp).float().cpu()
            for j in range(q_out.shape[0]):
                all_q_embs.append(q_out[j])
    print(f"✅ {len(all_q_embs)} query embeddings cached.")

    all_fused_indices = list(range(len(fused_index)))

    print(f"\n{'='*70}")
    print(f" Training Config (8 improvements):")
    print(f"   GPU:             {gpu_name} ({vram_gb:.0f}GB VRAM)")
    print(f"   Temperature:     {TEMPERATURE}")
    print(f"   Random negs:     {NUM_RANDOM_NEGS}")
    print(f"   Grad accumulate: {ACCUMULATION_STEPS} (eff batch={TRAIN_BATCH*ACCUMULATION_STEPS})")
    print(f"   Lambda L1 reg:   {LAMBDA_REG}")
    print(f"   Val ratio:       {VAL_RATIO}")
    print(f"{'='*70}\n")

    for epoch in range(start_epoch, NUM_EPOCHS):
        trial_head.train()
        order = list(train_indices)
        random.shuffle(order)
        epoch_ce = epoch_reg = epoch_uniform = epoch_gate_peak = num_steps = 0
        optimizer.zero_grad()

        for s in range(0, len(order), TRAIN_BATCH):
            batch_idx = order[s:s+TRAIN_BATCH]
            B = len(batch_idx)

            all_pos = set(idx for bi in batch_idx for idx in qa_pairs[bi]['gt_indices'])
            hard_negs = set(hn for bi in batch_idx for hn in hard_neg_map.get(bi, []) if hn not in all_pos)
            neg_pool = [x for x in all_fused_indices if x not in all_pos and x not in hard_negs]
            rand_negs = random.sample(neg_pool, min(NUM_RANDOM_NEGS, len(neg_pool)))
            candidates = sorted(all_pos) + sorted(hard_negs) + rand_negs
            if not candidates:
                continue

            uniform_ce = math.log(len(candidates))
            candidate_embs = [fused_index[c].to(device=device, dtype=torch.float32) for c in candidates]
            idx2pos = {idx: p for p, idx in enumerate(candidates)}

            q_list = [all_q_embs[bi] for bi in batch_idx]
            batch_q = pad_query_batch(q_list).to(device=device, dtype=torch.float32)
            scores, w_q = trial_head(batch_q, candidate_embs)

            # Fix 2: Temperature
            scaled_scores = scores / TEMPERATURE

            ce_loss = torch.tensor(0.0, device=device)
            valid_q = 0
            for b in range(B):
                qi = batch_idx[b]
                pp = [idx2pos[g] for g in qa_pairs[qi]['gt_indices'] if g in idx2pos]
                if not pp:
                    continue
                ce_loss += -(scaled_scores[b, pp].max() - torch.logsumexp(scaled_scores[b], dim=0))
                valid_q += 1
            if valid_q > 0:
                ce_loss = ce_loss / valid_q

            reg_loss = trial_head.get_l1_reg(w_q)
            total_loss = ce_loss + LAMBDA_REG * reg_loss

            # Fix 3: Gradient accumulation
            total_loss = total_loss / ACCUMULATION_STEPS
            total_loss.backward()

            step_in_accum = (s // TRAIN_BATCH) + 1
            if step_in_accum % ACCUMULATION_STEPS == 0 or (s + TRAIN_BATCH) >= len(order):
                torch.nn.utils.clip_grad_norm_(trial_head.parameters(), GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad()

            epoch_ce += ce_loss.item()
            epoch_reg += reg_loss.item()
            epoch_uniform += uniform_ce
            epoch_gate_peak += w_q.max(dim=-1).values.mean().item()
            num_steps += 1

        scheduler.step()
        avg_ce = epoch_ce / max(num_steps, 1)
        avg_reg = epoch_reg / max(num_steps, 1)
        avg_uniform = epoch_uniform / max(num_steps, 1)
        avg_gate_peak = epoch_gate_peak / max(num_steps, 1)
        lr_now = scheduler.get_last_lr()[0]

        # Fix 6: Val eval
        val_r1 = evaluate_val(
            trial_head, val_indices, all_q_embs, fused_index,
            qa_pairs, hard_neg_map, all_fused_indices, device,
        )

        marker = ""
        if val_r1 > best_val_recall:
            best_val_recall = val_r1
            torch.save(trial_head.state_dict(), TRIAL_WEIGHTS_PATH)
            marker = " ★ BEST"

        torch.save({
            'epoch': epoch,
            'model': trial_head.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_recall': best_val_recall,
        }, TRIAL_CKPT_PATH)

        print(
            f"  Epoch {epoch+1:2d}/{NUM_EPOCHS}"
            f" | CE: {avg_ce:.4f} (unif~{avg_uniform:.2f})"
            f" | ValR@1: {val_r1:.4f}"
            f" | Gate↑: {avg_gate_peak:.3f}"
            f" | Reg: {avg_reg:.4f}"
            f" | LR: {lr_now:.1e}"
            f"{marker}"
        )

    if os.path.exists(TRIAL_CKPT_PATH):
        os.remove(TRIAL_CKPT_PATH)
    trial_head.load_state_dict(torch.load(TRIAL_WEIGHTS_PATH, map_location=device, weights_only=True))
    trial_head.eval()
    print(f"\n✅ Training done! Best Val R@1: {best_val_recall:.4f}")

print(f"\n{'='*60}")
print(f" OUTPUT: {TRIAL_WEIGHTS_PATH}")
print(" → Tải về từ Kaggle Output tab")
print(" → Upload lên Kaggle Dataset (vd: trial-head-weights)")
print(" → Add dataset đó vào connect.ipynb để eval")
print(f"{'='*60}")